# Verification


#### What the Pipeline Run Looks Like in the UI
From the pipeline lineage graph, here's what actually happened on the live run:
| Table | Output Records | Refresh Mode | Expectations | Notes |
|---|---|---|---|---|
| `stock_prices` | **67K** | Incremental | 5 met | Only processes new bronze data since last run |
| `macro_indicators` | **7.4K** | Incremental | 2 met, 1 unmet (205 rows dropped) | One expectation flagging issues |
| `macro_daily_wide` | **1.9K** | **Full recompute** | — | Must recompute because forward-fill is order-dependent |


**Why is `macro_daily_wide` a full recompute?** Forward-fill carries values forward through time. Adding one new row could change the "last known value" used for many future rows, so the engine recomputes the whole table instead of trying to do it incrementally. This is the correct behavior and incremental forward-fill is mathematically unsafe. 

**Why `stock_prices` and `macro_indicators` are incremental:** Their logic is row-by-row (cast, round, dedupe within a `(symbol, date)` partition). New bronze rows can be processed independently without touching old ones. 


**What this tells you:** 205 macro rows had null values. This is **expected and correct** — recall from the FRED data that VIX, treasuries, and the USD index aren't published on weekends/holidays, but FRED still returns those date rows with null. The `valid_value` expectation cleanly drops them so downstream tables only see real observations. This is the data quality framework doing exactly what it should.

In [0]:
%sql
SELECT symbol, trade_date, close, volume
FROM raghavan_trading_signals.silver.stock_prices
WHERE symbol = 'AAPL'
ORDER BY trade_date DESC
LIMIT 10;

In [0]:
%sql
SELECT *
FROM raghavan_trading_signals.silver.macro_daily_wide
ORDER BY indicator_date DESC
LIMIT 10;

In [0]:
%sql
SELECT COUNT(*) AS anomalies
FROM raghavan_trading_signals.silver.stock_prices
WHERE high < low;

In [0]:
%sql
SELECT *
FROM raghavan_trading_signals.silver.macro_indicators
LIMIT 10;